# Cobblemon Atlas｜Google Colab 批次渲染與 CPU/GPU 比較

此 Notebook 使用現有 Python 渲染器完成 **CPU WebP 實際渲染**，並另行測量 **CUDA 光柵化數學原型**。兩者的時間不可直接當成完整 CPU/GPU 渲染速度比。

首先將下列檔案上傳到 Google Drive `Cobblemon_Atlas`：
- `source_packs/development_resource_packs.zip`（壓縮檔內最外層需包含同名資料夾）
- `source_packs/merged_pack.zip`（壓縮檔內最外層需包含同名資料夾）
- `source_packs/render_portraits.py`（Mac 原始核心，不放公開 GitHub）
- `manifests/animation-actions.json`（Mac 完整掃描產物）

請在你信任的 Notebook 中授權 Google Drive；掛載後 Notebook 可存取該帳號雲端硬碟中的其他檔案。預設只測試少量動作，不會一次渲染全部候選。

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, zipfile, time
DRIVE_ROOT = Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT = Path('/content/cobblemon-bedrock')
SITE_ROOT = Path('/content/cobblemon-atlas')
BRANCH = 'feat/animation-manifest-scanner-phase1'
# 修復批次：只重新測試洛奇亞先前失敗的兩個空中動作。
FORMS = ['0249_lugia']
ACTIONS = ['air_idle', 'air_fly']
FPS, SECONDS, SIZE, SS = 12, 2, 224, 2
print('Requested forms:', FORMS, 'actions:', ACTIONS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
# 渲染前先驗證每個動作的來源與動態骨骼；失敗就停止，不浪費 GPU/CPU 配額。
import importlib.util
tool_path=SITE_ROOT/'tools/render_action_jobs.py'
spec=importlib.util.spec_from_file_location('atlas_jobs_validation',tool_path)
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
selected=mod.choose_actions(data['entries'],set(FORMS),ACTIONS)
expected={(f,a) for f in FORMS for a in ACTIONS}
actual={(e['form_id'],output_id) for e,action,output_id in selected}
assert actual==expected, f'Manifest 找不到指定動作：{expected-actual}'
diagnostics=[]
for entry,action,output_id in selected:
    row={'form':entry['form_id'],'action':output_id,'clip_key':action['clip_key'],'manifest_id':action['id'],'source_pack':action['source_pack'],'dynamic_bones':action['dynamic_bones'],'model_bones_matched':action['model_bones_matched']}
    diagnostics.append(row);print(row)
    assert action['model_bones_matched']>0 and action['dynamic_bones']>0, f'不允許渲染無動態骨骼候選：{row}'
    if entry['form_id']=='0249_lugia' and output_id in ('air_idle','air_fly'):
        assert action['source_pack']=='merged_pack', f'洛奇亞動作仍選錯來源：{row}'
selection_report=DRIVE_ROOT/'reports/source-selection-lugia.json'
selection_report.write_text(json.dumps({'git_commit':sha,'selected':diagnostics},ensure_ascii=False,indent=2))
print('來源確認通過，已同步診斷：',selection_report)
cmd=[sys.executable,str(tool_path),'--repo',str(WORK_ROOT),'--manifest',str(manifest),'--forms',*FORMS,'--actions',*ACTIONS,'--dry-run']
subprocess.run(cmd,check=True)

In [ ]:
# 只執行上面已通過來源檢查的洛奇亞修復批次；每個 job 完成後同步 Drive。
out=WORK_ROOT/'captures/atlas_action_pilot'
cmd=[sys.executable,str(SITE_ROOT/'tools/render_action_jobs.py'),'--repo',str(WORK_ROOT),'--manifest',str(manifest),'--forms',*FORMS,'--actions',*ACTIONS,'--fps',str(FPS),'--seconds',str(SECONDS),'--size',str(SIZE),'--ss',str(SS),'--out',str(out),'--sync-root',str(DRIVE_ROOT)]
started=time.perf_counter()
result=subprocess.run(cmd,check=False)
print('CPU pilot elapsed seconds:',round(time.perf_counter()-started,2),'exit code:',result.returncode)
print('Drive output:',DRIVE_ROOT/'renders')
print('Checkpoint:',DRIVE_ROOT/'checkpoints/render-jobs.json')

In [ ]:
# GPU 數學微基準：不是完整渲染器的 GPU 版本。GPU 執行階段才會有 CUDA 結果。
report=DRIVE_ROOT/'reports/raster-cpu-gpu-benchmark.json'
subprocess.run([sys.executable,str(SITE_ROOT/'tools/benchmark_raster_gpu.py'),'--triangles','32','--size',str(SIZE),'--repeats','5','--output',str(report)],check=True)
print('GPU benchmark:',report)

In [ ]:
# 檢查已保存的成果與錯誤；分享這份報表即可讓 ChatGPT 追蹤驗收。
checkpoint=DRIVE_ROOT/'checkpoints/render-jobs.json'
if checkpoint.is_file():
    report=json.loads(checkpoint.read_text())
    from collections import Counter
    print('Job status:',dict(Counter(v.get('status') for v in report['jobs'].values())))
    for job,record in report['jobs'].items(): print(job,record.get('status'),record.get('reason'))
else: print('尚未產生渲染工作報表')
print('Drive folder:',DRIVE_ROOT)